In [43]:
import boto3
import json
import uuid
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from botocore.exceptions import ClientError

In [44]:
#CONFIG
REGION = "us-east-2"
RAW_BUCKET = "kaggle-arxiv-dataset"
RAW_KEY = "arxiv-metadata-oai-snapshot.json"
VECTOR_BUCKET = "kaggle-arxiv-vector-bucket"
INDEX_NAME = "arxiv-index"
DYNAMO_TABLE = "arxiv_papers"

#TUNING
MAX_WORKERS = 10
VECTOR_BATCH_SIZE = 500  # Batch size for Vector Store
DYNAMO_BATCH_SIZE = 25   # Max batch size allowed by DynamoDB is 25

In [45]:
# CLIENTS 
s3 = boto3.client("s3", region_name=REGION)
bedrock = boto3.client("bedrock-runtime", region_name=REGION)
s3vectors = boto3.client("s3vectors", region_name=REGION)
dynamodb = boto3.resource("dynamodb", region_name=REGION)
table = dynamodb.Table(DYNAMO_TABLE)

In [46]:
# EMBEDDING FUNCTION
def get_titan_embedding(text):
    try:
        payload = {"inputText": text}
        response = bedrock.invoke_model(
            modelId="amazon.titan-embed-text-v2:0", body=json.dumps(payload)
        )
        result = json.loads(response['body'].read())
        return result["embedding"]

    except Exception as e:
        return None

In [47]:
# STORE TO DYNAMODB
def batch_write_to_dynamo(items):
    """
    Writes a batch of up to 25 items to DynamoDB.
    Handles 'UnprocessedItems' automatically.
    """
    try:
        with table.batch_writer() as batch:
            for item in items:
                batch.put_item(Item=item)
    except Exception as e:
        print(f"DynamoDB Write Error: {e}")

In [48]:
# GENERATE PAYLOAD FOR S3VECTOR AND DYNAMODB
def process_paper_logic(paper):
    """Generates the Embedding AND the Metadata Payload."""
    # skip paper if abstract isn't available
    if not paper.get("abstract"):
        return None
    
    # use paper id as primary id
    pid = str(paper.get("id")) if paper.get("id") else str(uuid.uuid4())
    
    # get embedding of the paper abstract
    emb = get_titan_embedding(paper["abstract"])
    
    if not emb:
        return None

    # prepare DYNAMODB Payload 
    dynamo_record = {
        "paper_id": pid,
        "title": paper.get("title", "Unknown"),
        "abstract": paper.get("abstract", ""),
        "authors": str(paper.get("authors_parsed", [])), 
        "date": paper.get("update_date", ""),
        "categories": paper.get("categories", ""),
        "doi": paper.get("doi", "")
    }

    # prepare S3VECTOR Payload
    # metadata contains category and year for filterig
    vector_record = {
        "key": pid,
        "data": {"float32": emb},
        "metadata": {
            "category": paper.get("categories", "unknown").split(" ")[0],
            "year": paper.get("update_date", "0000")[:4]
        }
    }

    return vector_record, dynamo_record

In [50]:
MAX_TEST_RECORDS = 1000  # Test Pipeline

print(f"Starting Data Ingestion Pipeline (Limit: {MAX_TEST_RECORDS} papers)...")

start_time = time.time()
response = s3.get_object(Bucket=RAW_BUCKET, Key=RAW_KEY)

vector_buffer = []
dynamo_buffer = []
total_processed = 0  # Global counter

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    source = response['Body'].iter_lines()
    futures = []
    
    pbar = tqdm(total=MAX_TEST_RECORDS, desc="Processing", unit=" papers")

    while total_processed < MAX_TEST_RECORDS:
        
        # 1. Fill Thread Queue
        # We check both the Batch Size AND the Global Limit
        while len(futures) < VECTOR_BATCH_SIZE:
            try:
                # If we have already submitted enough tasks to hit the limit, stop reading
                if (total_processed + len(futures)) >= MAX_TEST_RECORDS:
                    break
                
                line = next(source)
                if not line:
                    break
                
                # Submit task
                futures.append(executor.submit(process_paper_logic, json.loads(line)))
            
            except StopIteration:
                break
        
        if not futures:
            break

        # 2. Collect Results
        for future in as_completed(futures):
            res = future.result()
            if res:
                v_rec, d_rec = res
                vector_buffer.append(v_rec)
                dynamo_buffer.append(d_rec)

                total_processed += 1
                pbar.update(1)

        # 3. UPLOADS (Batched)
        # A. Upload Vectors
        if len(vector_buffer) >= VECTOR_BATCH_SIZE:
            try:
                s3vectors.put_vectors(
                    vectorBucketName=VECTOR_BUCKET,
                    indexName=INDEX_NAME,
                    vectors=vector_buffer
                )
                # Clear buffer
                vector_buffer = []
            except Exception as e:
                print(f"   x Vector Upload Fail: {e}")

        # B. Upload Text to DynamoDB
        while len(dynamo_buffer) >= DYNAMO_BATCH_SIZE:
            batch = dynamo_buffer[:DYNAMO_BATCH_SIZE]
            dynamo_buffer = dynamo_buffer[DYNAMO_BATCH_SIZE:]
            batch_write_to_dynamo(batch)

        # Reset futures for next loop
        futures = []

    # FINAL FLUSH 
    if vector_buffer:
        s3vectors.put_vectors(vectorBucketName=VECTOR_BUCKET, indexName=INDEX_NAME, vectors=vector_buffer)

    if dynamo_buffer:
        # Process remaining dynamo items in chunks of 25
        for i in range(0, len(dynamo_buffer), DYNAMO_BATCH_SIZE):
            batch = dynamo_buffer[i:i+DYNAMO_BATCH_SIZE]
            batch_write_to_dynamo(batch)

pbar.close()

end_time = time.time()
duration = end_time - start_time

print(f"\nPipeline Test Complete. Processed {total_processed} papers.")
print(f"Total Ingestion Time: {duration:.2f} seconds")
print(f"Speed: {total_processed / duration:.2f} papers/second")


# print("All embeddings uploaded successfully to S3 Vector index.")
# print("Pipeline Complete.")

Starting Data Ingestion Pipeline (Limit: 1000 papers)...


Processing: 100%|██████████| 1000/1000 [00:47<00:00, 21.00 papers/s]


Pipeline Test Complete. Processed 1000 papers.
Total Ingestion Time: 47.73 seconds
Speed: 20.95 papers/second
